# 09 Stage 2 Logistic Regression — Health Outcome Prediction

**Owner:** PBC  
**Targets:** `target_unmet_fp` (and `target_anc_gap` when m14 is available)  
**Depends on:** `07_data_integration.ipynb`, `08_clustering.ipynb`

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.stage2_logistic import configure_logging, train_all_stage2_logistic
from src.models.stage2_xgboost import STAGE2_TARGETS, TARGET_DISPLAY, load_stage2_data

configure_logging()

In [2]:
# Ensure Stage 2 inputs exist
stage2_files = [
    PROJECT_ROOT / 'data/processed/stage2/X_stage2_preclustering.csv',
    PROJECT_ROOT / 'data/processed/stage2/y_stage2_targets.csv',
    PROJECT_ROOT / 'outputs/stage2_results/cluster_assignments.csv',
]
if not all(p.exists() for p in stage2_files):
    raise FileNotFoundError('Run scripts/run_stage2_data_prep.py or 08_clustering.ipynb first.')
print('Stage 2 inputs found.')

Stage 2 inputs found.


In [3]:
X_full, y = load_stage2_data()
print(f'Feature matrix (with cluster dummies): {X_full.shape}')
for col in y.columns:
    nn = y[col].notna().sum()
    pos = y.loc[y[col].notna(), col].sum() if nn else 0
    print(f'{col}: non-null N = {nn:,} | positive = {int(pos):,}')

2026-07-25 22:09:44,877 | INFO | src.models.stage2_xgboost | Loaded Stage 2 data: X=(724115, 43), targets=['target_unmet_fp', 'target_anc_gap']


Feature matrix (with cluster dummies): (724115, 43)
target_unmet_fp: non-null N = 466,859 | positive = 49,672
target_anc_gap: non-null N = 0 | positive = 0


In [4]:
# Train separate LogisticRegression models per target (Section 6.2)
results = train_all_stage2_logistic()
metrics_df = results.get('metrics_df')
if metrics_df is not None:
    display_cols = ['Target', 'TrainSize', 'TestSize', 'ROC-AUC', 'F1-Score', 'CV_ROC-AUC', 'Barrier_Uplift']
    metrics_df[[c for c in display_cols if c in metrics_df.columns]]

2026-07-25 22:09:52,089 | INFO | src.models.stage2_xgboost | Loaded Stage 2 data: X=(724115, 43), targets=['target_unmet_fp', 'target_anc_gap']
2026-07-25 22:09:52,094 | WARNING | src.models.stage2_logistic | Skipping target_anc_gap: No non-null rows for target_anc_gap. If target_anc_gap, ensure m14 is present in the raw extract.
2026-07-25 22:09:52,362 | INFO | src.models.stage2_xgboost | target_unmet_fp — analytic sample: 466859 rows (positive rate 0.1064)


--- Top predictors for target_unmet_fp ---
               Feature  Coefficient  OddsRatio
          v501_married     0.297631   1.346664
         v743f_missing     0.194428   1.214616
household_barrier_prob     0.140163   1.150461
                  v106     0.134574   1.144049
      v717_not working     0.100740   1.105989
        v130_christian     0.080020   1.083309
                  v013     0.077444   1.080522
   vulnerability_score     0.053735   1.055205
            v131_tribe     0.044319   1.045316
           v130_muslim     0.041564   1.042440

=== Logistic Regression | target_unmet_fp barrier ===
  Model       : Logistic Regression
  Target      : target_unmet_fp
  Accuracy    : 0.5847
  ROC-AUC     : 0.6591
  Precision   : 0.1573
  Recall      : 0.6665
  F1-Score    : 0.2546
              precision    recall  f1-score   support

           0       0.94      0.58      0.71     83438
           1       0.16      0.67      0.25      9934

    accuracy                          

2026-07-25 22:14:02,788 | INFO | src.models.stage2_logistic | Saved evaluation metrics -> C:\major project\BarrierLens_MP_G25_P48\outputs\stage2_results\logistic_evaluation_results.csv


target_unmet_fp: socioeconomic-only=0.6566 | +barriers=0.6583 | uplift=+0.0017


In [5]:
# Top odds-ratio predictors per target
for target_col in STAGE2_TARGETS:
    if target_col not in results.get('targets', {}):
        print(f'Skipped {TARGET_DISPLAY.get(target_col, target_col)} (no analytic sample)')
        continue
    coefs = results['targets'][target_col]['coefficients']
    print(f'\n=== {TARGET_DISPLAY.get(target_col, target_col)} — top odds ratios ===')
    print(coefs.head(10).to_string(index=False))

Skipped ANC Care Gap (no analytic sample)

=== Unmet Family Planning Need — top odds ratios ===
               Feature  Coefficient  OddsRatio
          v501_married     0.297631   1.346664
         v743f_missing     0.194428   1.214616
household_barrier_prob     0.140163   1.150461
                  v106     0.134574   1.144049
      v717_not working     0.100740   1.105989
        v130_christian     0.080020   1.083309
                  v013     0.077444   1.080522
   vulnerability_score     0.053735   1.055205
            v131_tribe     0.044319   1.045316
           v130_muslim     0.041564   1.042440
